# ResDMD residuals and finite-matrix pseudospectra

This notebook demonstrates **residual dynamic mode decomposition (ResDMD)** on a
**fixed** identity dictionary and a **finite-matrix** resolvent-norm grid on the
resulting EDMD operator. You will assemble consecutive observables, call
`resdmd`, plot Colbrook–Townsend residuals, and inspect
`resolvent_norm_grid` / `empirical_spectral_measure` on a tiny linear map.

**Honesty banner.** ResDMD here is **not**
`koopman_graph.analysis.spectral_residuals`. That diagnostic lives in the
**learned** observable norm after encode → Koopman → decode and is covered in
[`07_koopman_spectrum.ipynb`](07_koopman_spectrum.ipynb). This notebook uses
Galerkin Grams on a fixed dictionary (Colbrook & Townsend, *Commun. Pure Appl.
Math.*, online 2023 / print 2024; Colbrook, Ayton & Szőke, *J. Fluid Mech.*
2023). The resolvent grid is a **finite-matrix** numerical tool — not a
certified infinite-dimensional pseudospectrum or continuous spectral measure.


## Setup

Imports, seeds, and a non-interactive Matplotlib backend under pytest / CI.


In [ ]:
import os
import random
import warnings

from tqdm.std import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import matplotlib

if os.environ.get("PYTEST_CURRENT_TEST") or os.environ.get("CI"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import torch

from koopman_graph.analysis import (
    empirical_spectral_measure,
    resdmd,
    resolvent_norm_grid,
)
from koopman_graph.analysis._galerkin import (
    assemble_edmd_matrix,
    assemble_galerkin_grams,
)

SEED = 40
random.seed(SEED)
torch.manual_seed(SEED)

print("torch", torch.__version__)


## Motivation and background

Extended DMD (EDMD) fits a linear map on a **fixed** dictionary of observables.
ResDMD (Colbrook & Townsend, 2023/2024; Colbrook et al., 2023) adds the Galerkin
matrix for $\mathcal{K}^*\mathcal{K}$ so each candidate eigenpair $(\lambda, g)$
carries a residual that, in the large-data limit, approximates the
infinite-dimensional Koopman residual. Sample Grams on consecutive lifts
$\Psi_0, \Psi_1$ are

$$
G_{00} = \Psi_0^H \Psi_0,\quad
G_{01} = \Psi_0^H \Psi_1,\quad
G_{11} = \Psi_1^H \Psi_1.
$$

By contrast, `spectral_residuals` in notebook 07 projects **learned** latents
onto eigenmodes of a trained model. Mixing those names hides whether you are
certifying a fixed dictionary or diagnosing a neural observable map.

Separately, the $\varepsilon$-pseudospectrum of a matrix $A$ can be probed via
the resolvent 2-norm $\|(zI - A)^{-1}\|_2 = 1 / \sigma_{\min}(zI - A)$. Here we
evaluate that on a coarse complex-plane grid for the finite EDMD matrix only.


## Minimal example: identity dictionary on a linear map

Take $\psi(x) = x$ and hand-built pairs $\Psi_0 = I_2$, $\Psi_1 = A$ with
$A = \mathrm{diag}(0.5, 0.2)$ (row map $x \mapsto x A$). Then $G_{00} = I$ and
true EDMD eigenpairs have ResDMD residual approximately zero; a wrong
$(\lambda, g)$ recovers the analytic bound
$\|(A - \lambda I)g\|_2 / \|g\|_2$ (here $0.5$ for $\lambda = 0$, $g = e_1$).


In [ ]:
a = torch.diag(torch.tensor([0.5, 0.2], dtype=torch.float64))
psi0 = torch.eye(2, dtype=torch.float64)
psi1 = a.clone()

report = resdmd(psi0, psi1, tolerance=1e-3)
print("eigenvalues:", report.eigenvalues)
print("residuals:", report.residuals)
print("trustworthy:", report.trustworthy_mask())
assert torch.all(report.residuals < 1e-8)

grams = assemble_galerkin_grams(psi0, psi1)
edmd = assemble_edmd_matrix(grams)
print("EDMD matrix:\n", edmd)


## Progressive deep dive: resolvent grid and spectral measure

`resolvent_norm_grid` evaluates $\|(zI - A)^{-1}\|_2$ on a rectangular grid in
the complex plane (`resolvent_norms` has shape `(n_imag, n_real)`).
`empirical_spectral_measure` places uniform weights $1/k$ on the $k$
eigenvalues of the same finite matrix — a discrete point measure, not a
certified continuous spectral measure of an infinite-dimensional Koopman
operator. Grid extent and resolution trade cost ($O(n_{\mathrm{imag}}\,
n_{\mathrm{real}}\, m^3)$) against visual detail; keep both small for smoke runs.


In [ ]:
real_grid = torch.linspace(-1.0, 1.0, 9, dtype=torch.float64)
imag_grid = torch.linspace(-0.5, 0.5, 5, dtype=torch.float64)
grid = resolvent_norm_grid(edmd, real_grid, imag_grid)
measure = empirical_spectral_measure(operator=edmd)

print("resolvent_norms shape:", tuple(grid.resolvent_norms.shape))
print("measure eigenvalues:", measure.eigenvalues)
print("weights sum:", float(measure.weights.sum()))
assert grid.resolvent_norms.shape == (5, 9)
assert abs(float(measure.weights.sum()) - 1.0) < 1e-12


## Results

The left panel marks EDMD eigenvalues in the complex plane. The right panel
shows $\log_{10}$ of the resolvent 2-norm: look for **bright peaks near the
eigenvalues** (small $\sigma_{\min}$) and lower values far from the spectrum.


In [ ]:
eigs = measure.eigenvalues.detach().cpu()
log_norm = torch.log10(grid.resolvent_norms).detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))

axes[0].scatter(eigs.real, eigs.imag, c="C0", s=60, zorder=3)
axes[0].axhline(0.0, color="0.7", lw=0.8)
axes[0].axvline(0.0, color="0.7", lw=0.8)
axes[0].set_xlabel("Re($\\lambda$)")
axes[0].set_ylabel("Im($\\lambda$)")
axes[0].set_title("EDMD eigenvalues")
axes[0].set_aspect("equal", adjustable="datalim")

extent = [
    float(real_grid[0]),
    float(real_grid[-1]),
    float(imag_grid[0]),
    float(imag_grid[-1]),
]
im = axes[1].imshow(
    log_norm,
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap="magma",
)
axes[1].scatter(eigs.real, eigs.imag, c="cyan", s=40, edgecolors="k", lw=0.5)
axes[1].set_xlabel("Re($z$)")
axes[1].set_ylabel("Im($z$)")
axes[1].set_title("$\\log_{10} \\|(zI-A)^{-1}\\|_2$")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()


## Interpretation / discussion

On this exact linear identity-dictionary fixture, ResDMD residuals collapse to
numerical noise for true eigenpairs — as expected when the dictionary spans the
dynamics. The resolvent heatmap is consistent with that spectrum: norms inflate
near eigenvalues. For learned graph Koopman models, prefer notebook 07’s
`spectral_residuals` as a **held-out diagnostic**; use `resdmd` when you have a
**fixed** lift (identity, polynomial EDMD, frozen encodings) and want
Colbrook–Townsend-style residual bounds on that dictionary. Neither plot here
certifies infinite-dimensional pseudospectra or continuous spectral measures.


## Takeaways

- Use `resdmd` on pre-lifted tensor pairs `(psi0, psi1)` for fixed-dictionary
  Colbrook–Townsend residuals; do **not** rename or treat
  `spectral_residuals` as ResDMD.
- `resolvent_norm_grid` / `empirical_spectral_measure` act on a **square
  assembled matrix** (EDMD $A$, dense $K$, or $K_{\mathrm{eff}}$) — finite-matrix
  MVP only.
- Identity-dictionary linear maps are a useful oracle: true modes → near-zero
  residual; wrong $(\lambda, g)$ → known $\|(A-\lambda I)g\|/\|g\|$.
- Keep grids coarse in CI / smoke runs; cost scales with grid size and $m^3$.
- For learned-operator spectrum diagnostics after training, start from
  [`07_koopman_spectrum.ipynb`](07_koopman_spectrum.ipynb).


## Further reading

- Colbrook, M. J. & Townsend, A. (2023/2024). Rigorous data-driven computation
  of spectral properties of Koopman operators for dynamical systems.
  *Communications on Pure and Applied Mathematics*, 77(1), 221–283.
  https://doi.org/10.1002/cpa.22125 (`ColbrookTownsend2023ResDMD`)
- Colbrook, M. J., Ayton, L. J. & Szőke, M. (2023). Residual dynamic mode
  decomposition: robust and verified Koopmanism. *Journal of Fluid Mechanics*,
  955, A21. https://doi.org/10.1017/jfm.2022.1052 (`Colbrook2023ResidualDMD`)
- Related tutorial: [`07_koopman_spectrum.ipynb`](07_koopman_spectrum.ipynb)
  (learned spectrum + `spectral_residuals`)
- API: `koopman_graph.analysis.resdmd`,
  `koopman_graph.analysis.resolvent_norm_grid`,
  `koopman_graph.analysis.empirical_spectral_measure`
